# PASE in a few minutes: real GSE234933 myeloid cells

This tutorial uses **776 genuine cells from eight donors**, selected deterministically from the public GSE234933 count files. It runs offline after the package and notebook dependencies are installed. The bundled data are small enough for an ordinary laptop.

This is a software tutorial, **not a reproduction of the full-cohort manuscript analysis** or a new validation cohort. The 15-gene outcome below illustrates the API's within-patient standardized score; it is not claimed to duplicate a published scoring implementation.

Before opening the notebook, install the software and notebook extra using the installation command in the repository README. No large GEO download is needed for this tutorial. Rebuilding the fixture from the approximately 1 GB original archive is a separate optional task described at the end.

In [ ]:
%matplotlib inline
from time import perf_counter
started = perf_counter()
import pandas as pd
import matplotlib.pyplot as plt
import pase
from pase.datasets import load_gse234933_demo

## 1. Inspect the input

`AnnData.X` contains log1p(CP10K). Its size-factor denominator was calculated across **all original genes**, before reducing the matrix to 19 teaching genes. `log_depth` is log1p of this full library size. The `counts` layer contains only the 19 retained genes, so its row sum must not be used to renormalize this subset.

The source authors' broad myeloid annotation supplies `myeloid`; donor labels and original cell barcodes are retained. No clinical labels have been invented. The extraction uses the first eight lexicographically sorted specimens, up to 100 cells each, selected by a seed-and-barcode hash without looking at correlations.

In [ ]:
adata = load_gse234933_demo()
print(adata)
adata.obs.groupby('donor', observed=True).agg(cells=('specimen', 'size'), median_library=('full_library_size', 'median'))

In [ ]:
target = adata.uns['dataset']['spp1_tam_genes']
print('Anchor: LILRB4')
print('Outcome genes:', ', '.join(target))
assert len(target) == 15 and set(target).issubset(adata.var_names)
adata.obs.head()

## 2. Estimate an equal-patient mean

For this multi-gene outcome, PASE standardizes each target gene within each patient's selected cells and averages the standardized values. It separately removes an intercept and `log_depth` from anchor and outcome, then correlates the two residual vectors within each patient.

`aggregate='equal'` gives each eligible patient one contribution to the arithmetic mean of the observed correlations. Its interval uses the Student t distribution across patients. `min_cells=30` is a chosen eligibility setting, not a universally validated guarantee of reliable inference. This small tutorial demonstrates computation; it is too small to establish calibration or a biological claim.

In [ ]:
equal = pase.associate(
    adata, anchor='LILRB4', target=target, patient_key='donor',
    covariates=['log_depth'], cell_mask='myeloid',
    aggregate='equal', min_cells=30,
)
equal.summary()

In [ ]:
equal.patients

The patient table retains eligible and excluded donors, including their reasons. `n_patients_total` and `n_patients_eligible` in the summary distinguish the input population from the patients contributing to the reported result.

In [ ]:
equal.plot_forest(title='GSE234933 teaching subset: equal-patient association')
plt.show()

## 3. Compare targets before comparing numbers

The random-effects option transforms each eligible correlation using Fisher z, approximates its sampling variance, fits between-patient heterogeneity by REML, and uses a classical HKSJ t interval. The back-transformed cohort mean is **not the arithmetic mean of patient correlations**. Its approximations can be unreliable for sparse count-derived scores or very small patient numbers.

The empirical Bayes option estimates individual patient effects conditional on fitted population parameters. Its intervals omit uncertainty from estimating those parameters, and it does not return a cohort P value or mean interval. These options answer different questions; their numerical differences are not, by themselves, evidence that one is better.

In the individual-effect plot, grey circles show the observed correlations; orange diamonds and bars show the EB estimates and their conditional 95% intervals.

In [ ]:
weighted = pase.associate(
    adata, anchor='LILRB4', target=target, patient_key='donor',
    covariates=['log_depth'], cell_mask='myeloid',
    aggregate='re_hksj', min_cells=30,
)
eb = pase.associate(
    adata, anchor='LILRB4', target=target, patient_key='donor',
    covariates=['log_depth'], cell_mask='myeloid',
    aggregate='eb', min_cells=30,
)
pd.concat([equal.summary(), weighted.summary(), eb.summary()], ignore_index=True)

In [ ]:
eb.plot_forest(title='GSE234933 teaching subset: conditional individual EB effects')
plt.show()

## Reporting policy

The default is six method-eligible patients. Below this floor, only descriptive results are returned. Reported inference with fewer than twenty eligible patients receives a small-cohort warning; a correlation-scale interval wider than 0.2 receives a configurable precision warning. Neither boundary guarantees calibration. Inspect `metadata["warnings"]`. An external planning SD is required for a model-based MDE; the tutorial does not infer power from its observed effects.


In [ ]:
sensitivity = {}
for threshold in [30, 80, 120]:
    sensitivity[threshold] = pase.associate(
        adata, anchor='LILRB4', target=target, patient_key='donor',
        covariates=['log_depth'], cell_mask='myeloid',
        aggregate='equal', min_cells=threshold,
    )
pd.concat([value.summary().assign(min_cells=key) for key, value in sensitivity.items()], ignore_index=True)

In [ ]:
sensitivity[120].patients

## 5. Use your own AnnData object

Provide cells by genes with unique gene names, an observed patient identifier in `obs`, and the transformed values you intend to analyze. `cell_mask` can name a Boolean observation column. Covariates must be measured cell-level numeric columns. PASE does not silently normalize `X`; full-library normalization belongs upstream. An optional `layer` selects another matrix of values already transformed for analysis.

For a four-gene API illustration, replace `target` with `['MMP14', 'MMP9', 'SPP1', 'APOE']`. That list is not the 15-gene endpoint used above. Always state the score, normalization, covariates, inclusion rules, and statistical target when reporting results.

## Source and optional rebuild

- Data: [GSE234933](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE234933).
- Original study: Bill et al., *Science* (2023), [10.1126/science.ade2292](https://doi.org/10.1126/science.ade2292).
- Fifteen-gene list: [10.3389/fimmu.2021.694801](https://doi.org/10.3389/fimmu.2021.694801). This tutorial uses PASE's own documented scoring rule.
- Rights: these third-party GEO-derived observations are not relicensed under PASE's MIT software license; see the bundled `DATA_NOTICE.md` and [GEO terms](https://www.ncbi.nlm.nih.gov/geo/info/disclaimer.html).

For a fresh reconstruction, download the original RNA count archive and cell-annotation file from GEO, extract the RDS files, install `rdata`, and run:

```bash
python examples/build_gse234933_subset.py SOURCE_DIR --output NEW_OUTPUT
```

The builder verifies exact source SHA-256 values, records original dimensions, computes full-library normalization, and writes output checksums. This optional download and reconstruction are outside the quick tutorial runtime.

In [ ]:
info = adata.uns['dataset']
print('Data source:', info['source_url'])
print('Source checksums:', len(info['source_file_sha256']))
print('Elapsed notebook computation: %.2f seconds' % (perf_counter() - started))